# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset is described with a Croissant schema available at:

[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Metadata object has fields as attributes, not as a dictionary
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

# Show the publication date and license
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Let's enumerate record sets. For each record set, we'll list fields (columns) and display their `@id` and names.

In [ ]:
# List available record sets and their fields using their @id
record_sets = dataset.metadata.recordSet
overview = []

for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    fields = rs.get('field', [])
    for field in fields:
        field_id = field.get('@id', '(no id)')
        fname = field.get('name', '(no name)')
        print(f"  Field: {field_id} | name: {fname}")
    overview.append({'record_set_id': rs['@id'], 'fields': [field.get('@id') for field in fields]})

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We'll use the record set and field `@id` values identified above. For demonstration, extract data from the main record set and display its columns.

In [ ]:
dataframes = {}

# Prepare a list of record set @id values
record_set_ids = [o['record_set_id'] for o in overview]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"DataFrame for {rs_id} loaded: {df.shape[0]} rows x {df.shape[1]} columns")

# Show columns for the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping.

Suppose we want to analyze the GAD-7 total score (`gad7_total_score`) field. We'll reference it by its column (field) `@id` in the schema.

In [ ]:
# Update this to the actual @id for GAD-7 score (field @id from overview step).
gad7_field_id = None
# Find a field @id for GAD-7, or fallback to 'gad7_total_score' if present
for f in overview[0]['fields']:
    if 'gad7' in f.lower():
        gad7_field_id = f
        break
if gad7_field_id is None:
    gad7_field_id = 'gad7_total_score'  # fallback if field name is literal

# Choose a reasonable threshold if scores are in the range 0-21
threshold = 10
df = dataframes[main_record_set_id]
if gad7_field_id in df.columns:
    filtered_df = df[df[gad7_field_id] > threshold]
    print(f"Filtered records with {gad7_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
    print(f"Normalized {gad7_field_id} for filtered records:")
    print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

    # Group by demographic field, e.g. 'level_of_education' or its @id
    group_field_id = None
    for f in overview[0]['fields']:
        if 'education' in f.lower():
            group_field_id = f
            break
    if group_field_id is None:
        group_field_id = 'level_of_education' # fallback

    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[gad7_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean GAD-7):")
        print(grouped_df.head())
else:
    print(f"Field '{gad7_field_id}' not found in main record set columns.")

## 5. Visualization
Visualize the distribution of GAD-7 scores and their relationship to education level.

In [ ]:
if gad7_field_id in df.columns:
    plt.figure(figsize=(8,4))
    df[gad7_field_id].hist(bins=15)
    plt.title('Distribution of GAD-7 Total Score')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        df.groupby(group_field_id)[gad7_field_id].mean().plot(kind='bar')
        plt.title(f'Mean GAD-7 Score by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel('Mean GAD-7 Score')
        plt.show()

## 6. Conclusion
This notebook demonstrated exploration of the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya, using `mlcroissant`. 

- We loaded metadata and records using the Croissant schema.
- We identified record sets and referenced all fields by their `@id`.
- We explored and visualized GAD-7 scores by educational level.

Further analysis could include deeper exploration of other assessment scores (PHQ-9, PSQ), demographic correlations, and more advanced preprocessing.

Please cite the dataset appropriately using its provided citation, and respect its open data license.